In [1]:
import zipfile
import pandas as pd
import zipfile
from pyproj import Transformer

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## **School Data**

In [ ]:
# read in schools from 2024-2025 academic year (last year in study)
schoolsForAnalysis = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/SecondarySchools/finalSecondarySchools.csv', encoding="latin-1", low_memory=False)

# read in schools from 2016-2017 academic year (first year in study)
schoolData20162017 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2016-2017/england_ks4final.csv', encoding='utf-8-sig', low_memory=False)

In [ ]:
# filter to same list of schools for the temporal analysis
schoolData20162017 = schoolData20162017[schoolData20162017['URN'].isin(schoolsForAnalysis['URN'])].copy()

finalSchoolList = schoolData20162017.copy()

# write to drive
finalSchoolList.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/SecondarySchools/finalSchoolListTemporal.csv", index=False)

In [ ]:
# read in schools from 2018-2019 academic year
schoolData20182019 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2018-2019/england_ks4final.csv', encoding="latin-1", low_memory=False)

# read in schools from 2020-2021 academic year
schoolData20212022 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2021-2022/england_ks4final.csv', encoding="latin-1", low_memory=False)

# read in schools from 2022-2023 academic year
schoolData20222023 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2022-2023/england_ks4final.csv', encoding="latin-1", low_memory=False)

# read in schools from 2024-2025 academic year
schoolData20242025 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2024-2025/england_ks4final.csv', encoding="latin-1", low_memory=False)

In [ ]:
# filter to same list of schools for the temporal analysis
schoolData20182019 = schoolData20182019[schoolData20182019['URN'].isin(finalSchoolList['URN'])].copy()

# filter to same list of schools for the temporal analysis
schoolData20212022 = schoolData20212022[schoolData20212022['URN'].isin(finalSchoolList['URN'])].copy()

# filter to same list of schools for the temporal analysis
schoolData20222023 = schoolData20222023[schoolData20222023['URN'].isin(finalSchoolList['URN'])].copy()

# filter to same list of schools for the temporal analysis
schoolData20242025 = schoolData20242025[schoolData20242025['URN'].isin(finalSchoolList['URN'])].copy()

In [ ]:
# clean the dataframes, remove whitespace, convert to integers, etc.
def clean_percent(series):
    return pd.to_numeric(
        series.astype(str).str.replace('%', '', regex=False).str.strip(),
        errors='coerce'
    )
# rename columns for independent/dependent variable titles
def standardise_school_data(df):
    cols = {
        'URN': 'URN',
        'ESTAB': 'EstablishmentNumber',
        'SCHNAME': 'EstablishmentName',
        'PTL2BASICS_95': 'StudentsAchievingGrade5PlusEngMathRate',
        'TPUP': 'NumberOfPupils',
        'TFSM6CLA1A': 'StudentReceivingFreeSchoolMeals',
        'PTFSM6CLA1A': 'PercentOfStudentReceivingFreeSchoolMeals',
        'TOTPUPS': 'TotalPupils',
        'NUMGIRLS': 'FemalePupils',
    }
    present = {k: v for k, v in cols.items() if k in df.columns}
    out = df[list(present.keys())].rename(columns=present).copy()

    for c in ['StudentsAchievingGrade5PlusEngMathRate', 'PercentOfStudentReceivingFreeSchoolMeals']:
        if c in out.columns:
            out[c] = clean_percent(out[c])

    for c in ['NumberOfPupils', 'StudentReceivingFreeSchoolMeals', 'TotalPupils', 'FemalePupils']:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce')

    # only compute if both columns are present
    if 'FemalePupils' in out.columns and 'TotalPupils' in out.columns:
      out['PercentFemale'] = (out['FemalePupils'] / out['TotalPupils'] * 100).round(2)

    return out

schoolData20162017 = standardise_school_data(schoolData20162017)
schoolData20182019 = standardise_school_data(schoolData20182019)
schoolData20212022 = standardise_school_data(schoolData20212022)
schoolData20222023 = standardise_school_data(schoolData20222023)
schoolData20242025 = standardise_school_data(schoolData20242025)

In [ ]:
schoolData20212022

,URN,EstablishmentNumber,EstablishmentName,StudentsAchievingGrade5PlusEngMathRate,NumberOfPupils,StudentReceivingFreeSchoolMeals,PercentOfStudentReceivingFreeSchoolMeals,TotalPupils,FemalePupils,PercentFemale
4,100053.0,4285.0,Acland Burghley School,53.0,163.0,69.0,42.0,1168,400,34.25
6,100054.0,4611.0,The Camden School for Girls,87.0,117.0,41.0,35.0,1069,907,84.85
10,100052.0,4275.0,Hampstead School,45.0,193.0,86.0,45.0,1311,624,47.60
11,100049.0,4104.0,Haverstock School,53.0,128.0,80.0,63.0,968,388,40.08
12,100059.0,5401.0,La Sainte Union Catholic Secondary School,63.0,168.0,53.0,32.0,909,877,96.48
...,...,...,...,...,...,...,...,...,...,...
5754,138235.0,4001.0,The Parker E-ACT Academy,47.0,188.0,49.0,26.0,1120,600,53.57
5760,139690.0,4011.0,Silverstone UTC,29.0,123.0,24.0,20.0,492,88,17.89
5761,136488.0,4004.0,Sponne School,69.0,225.0,19.0,8.0,1446,706,48.82
5762,142747.0,4703.0,Thomas Becket Catholic School,31.0,134.0,36.0,27.0,840,384,45.71


In [ ]:
len(schoolData20212022)

2285

## **Urban Rural Classification**

In [ ]:
# columns to bring in, with renaming
geo_cols = {
    'URN': 'URN',
    'UrbanRural (name)': 'UrbanRuralClassification',
    'Postcode': 'Postcode',
    'GOR (name)': 'GOR',
    'LA (code)': 'LA (code)',
    'LA (name)': 'LA (name)',
    'Easting': 'Easting',
    'Northing': 'Northing',
    'MSOA (code)': 'MSOA (code)',
    'MSOA (name)': 'MSOA (name)',
    'LSOA (code)': 'LSOA (code)',
}

geo = schoolsForAnalysis[list(geo_cols.keys())].rename(columns=geo_cols)

# merge onto each year
schoolData20162017 = schoolData20162017.merge(geo, on='URN', how='left')
schoolData20182019 = schoolData20182019.merge(geo, on='URN', how='left')
schoolData20212022 = schoolData20212022.merge(geo, on='URN', how='left')
schoolData20222023 = schoolData20222023.merge(geo, on='URN', how='left')
schoolData20242025 = schoolData20242025.merge(geo, on='URN', how='left')

In [ ]:
def add_urban_rural_dummies(df):
    df = df.copy()
    mapping = {
        'LargeCentralMetro': 'Urban: Nearer to a major town or city',
        'LargeFringeMetro':  'Urban: Further from a major town or city',
        'MediumMetro':       'Larger rural: Nearer to a major town or city',
        'SmallMetro':        'Larger rural: Further from a major town or city',
        'Micropolitan':      'Smaller rural: Nearer to a major town or city',
        'Noncore':           'Smaller rural: Further from a major town or city',
    }
    for dummy_name, category in mapping.items():
        df[dummy_name] = (df['UrbanRuralClassification'] == category).astype(int)
    return df

schoolData20162017 = add_urban_rural_dummies(schoolData20162017)
schoolData20182019 = add_urban_rural_dummies(schoolData20182019)
schoolData20212022 = add_urban_rural_dummies(schoolData20212022)
schoolData20222023 = add_urban_rural_dummies(schoolData20222023)
schoolData20242025 = add_urban_rural_dummies(schoolData20242025)

In [ ]:
schoolData20222023.head()

,URN,EstablishmentNumber,EstablishmentName,StudentsAchievingGrade5PlusEngMathRate,NumberOfPupils,StudentReceivingFreeSchoolMeals,PercentOfStudentReceivingFreeSchoolMeals,TotalPupils,FemalePupils,PercentFemale,...,Northing,MSOA (code),MSOA (name),LSOA (code),LargeCentralMetro,LargeFringeMetro,MediumMetro,SmallMetro,Micropolitan,Noncore
0,100053.0,4285.0,Acland Burghley School,57.0,178.0,65.0,37.0,1163,398.0,34.22,...,185931.0,E02000168,Camden 003,E01000928,1,0,0,0,0,0
1,100054.0,4611.0,The Camden School for Girls,77.0,116.0,42.0,36.0,1047,908.0,86.72,...,184659.0,E02000180,Camden 015,E01035707,1,0,0,0,0,0
2,100052.0,4275.0,Hampstead School,43.0,197.0,89.0,45.0,1319,638.0,48.37,...,185633.0,E02000170,Camden 005,E01000871,1,0,0,0,0,0
3,100049.0,4104.0,Haverstock School,43.0,151.0,95.0,63.0,982,423.0,43.08,...,184498.0,E02000177,Camden 012,E01000902,1,0,0,0,0,0
4,100059.0,5401.0,La Sainte Union Catholic Secondary School,49.0,152.0,62.0,41.0,817,777.0,95.10,...,186191.0,E02000168,Camden 003,E01000910,1,0,0,0,0,0


## **RACE**

In [ ]:
race = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/ethnicityCensus2021.csv', encoding="latin-1", low_memory=False)

In [ ]:
def broad_group(cat):
    cat = str(cat)
    if cat.startswith('Asian'):
        return 'Asian'
    elif cat.startswith('Black'):
        return 'Black'
    elif cat.startswith('White'):
        return 'White'
    elif cat.startswith('Mixed') or cat.startswith('Other ethnic group'):
        return 'Other'
    else:
        return 'Exclude'   # 'Does not apply'

race['BroadGroup'] = race['Ethnic group (20 categories)'].apply(broad_group)
race = race[race['BroadGroup'] != 'Exclude']

In [ ]:
lsoa_col = 'Lower layer Super Output Areas Code'

totals = race.groupby(lsoa_col)['Observation'].sum().rename('Total')

wide = race.pivot_table(
    index=lsoa_col, columns='BroadGroup',
    values='Observation', aggfunc='sum', fill_value=0
)

race_rates = wide.div(totals, axis=0) * 100
race_rates = race_rates.rename(columns={
    'Asian': 'AsianPopulationRate',
    'Black': 'BlackPopulationRate',
    'White': 'WhitePopulationRate',
    'Other': 'OtherRacePopulationRate',
}).reset_index().rename(columns={lsoa_col: 'LSOA (code)'})

race_cols = ['AsianPopulationRate', 'BlackPopulationRate',
             'WhitePopulationRate', 'OtherRacePopulationRate']

base = schoolData20242025.merge(
    race_rates[['LSOA (code)'] + race_cols],
    on='LSOA (code)', how='left'
)

race_msoa_lookup = base.groupby('MSOA (code)')[race_cols].mean().round(2).reset_index()

year_dfs = {
    '20162017': schoolData20162017,
    '20182019': schoolData20182019,
    '20212022': schoolData20212022,
    '20222023': schoolData20222023,
    '20242025': schoolData20242025,
}

for year, df in year_dfs.items():
    df = df.drop(columns=race_cols, errors='ignore')
    df = df.merge(race_msoa_lookup, on='MSOA (code)', how='left')
    year_dfs[year] = df

schoolData20162017 = year_dfs['20162017']
schoolData20182019 = year_dfs['20182019']
schoolData20212022 = year_dfs['20212022']
schoolData20222023 = year_dfs['20222023']
schoolData20242025 = year_dfs['20242025']

In [ ]:
schoolData20182019.head()

,URN,EstablishmentNumber,EstablishmentName,StudentsAchievingGrade5PlusEngMathRate,NumberOfPupils,StudentReceivingFreeSchoolMeals,PercentOfStudentReceivingFreeSchoolMeals,TotalPupils,FemalePupils,PercentFemale,...,LargeCentralMetro,LargeFringeMetro,MediumMetro,SmallMetro,Micropolitan,Noncore,AsianPopulationRate,BlackPopulationRate,WhitePopulationRate,OtherRacePopulationRate
0,100049.0,4104.0,Haverstock School,43.0,199.0,117.0,59.0,950,393.0,41.37,...,1,0,0,0,0,0,19.35,17.16,49.79,13.70
1,100050.0,4166.0,Parliament Hill School,58.0,175.0,94.0,54.0,1170,1123.0,95.98,...,1,0,0,0,0,0,8.82,10.26,66.75,14.17
2,100054.0,4611.0,The Camden School for Girls,75.0,107.0,38.0,36.0,1025,879.0,85.76,...,1,0,0,0,0,0,9.01,6.20,73.25,11.54
3,100056.0,4688.0,William Ellis School,41.0,111.0,58.0,52.0,836,NaN,NaN,...,1,0,0,0,0,0,8.82,10.26,66.75,14.17
4,137181.0,4000.0,The UCL Academy,47.0,171.0,100.0,59.0,1133,542.0,47.84,...,1,0,0,0,0,0,36.48,4.22,43.41,15.89


## **IDACI**

In [ ]:
idaci20162017 = pd.read_excel('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2016-2017/idaci.xlsx', sheet_name='ID2015 IDACI & IDAOPI')
idaci20182019 = pd.read_excel('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2018-2019/idaci.xlsx', sheet_name='IoD2019 IDACI & IDAOPI')
idaci20212022 = pd.read_excel('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2021-2022/idaci.xlsx', sheet_name='IoD2019 IDACI & IDAOPI')
idaci20222023 = pd.read_excel('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2022-2023/idaci.xlsx', sheet_name='IoD2025 IDACI & IDAOPI')
idaci20242025 = pd.read_excel('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2024-2025/idaci.xlsx', sheet_name='IoD2025 IDACI & IDAOPI')

In [ ]:
idaci20242025.columns.tolist()

['LSOA code (2021)',
 'LSOA name (2021)',
 'Local Authority District code (2024)',
 'Local Authority District name (2024)',
 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)',
 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)',
 'Income Deprivation Affecting Children Index (IDACI) Rank (where 1 is most deprived)',
 'Income Deprivation Affecting Children Index (IDACI) Decile (where 1 is most deprived 10% of LSOAs)',
 'Income Deprivation Affecting Older People (IDAOPI) Rank (where 1 is most deprived)',
 'Income Deprivation Affecting Older People (IDAOPI) Decile (where 1 is most deprived 10% of LSOAs)']

In [ ]:
lsoa2011to2022 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/Data/lsoa2011to2022.csv', encoding='utf-8-sig', low_memory=False)
schoolData20162017 = schoolData20162017.merge(lsoa2011to2022[['LSOA21CD', 'LSOA11CD']], left_on='LSOA (code)', right_on='LSOA21CD', how='left')
schoolData20182019 = schoolData20182019.merge(lsoa2011to2022[['LSOA21CD', 'LSOA11CD']], left_on='LSOA (code)', right_on='LSOA21CD', how='left')
schoolData20212022 = schoolData20212022.merge(lsoa2011to2022[['LSOA21CD', 'LSOA11CD']], left_on='LSOA (code)', right_on='LSOA21CD', how='left')
schoolData20222023 = schoolData20222023.merge(lsoa2011to2022[['LSOA21CD', 'LSOA11CD']], left_on='LSOA (code)', right_on='LSOA21CD', how='left')
schoolData20242025 = schoolData20242025.merge(lsoa2011to2022[['LSOA21CD', 'LSOA11CD']], left_on='LSOA (code)', right_on='LSOA21CD', how='left')

In [ ]:
schoolData20182019.head()

,URN,EstablishmentNumber,EstablishmentName,StudentsAchievingGrade5PlusEngMathRate,NumberOfPupils,StudentReceivingFreeSchoolMeals,PercentOfStudentReceivingFreeSchoolMeals,TotalPupils,FemalePupils,PercentFemale,...,Northing,MSOA (code),MSOA (name),LSOA (code),AsianPopulationRate,BlackPopulationRate,WhitePopulationRate,OtherRacePopulationRate,LSOA21CD,LSOA11CD
0,100049.0,4104.0,Haverstock School,43.0,199.0,117.0,59.0,950,393.0,41.37,...,184498.0,E02000177,Camden 012,E01000902,19.35,17.16,49.79,13.70,E01000902,E01000902
1,100050.0,4166.0,Parliament Hill School,58.0,175.0,94.0,54.0,1170,1123.0,95.98,...,186019.0,E02000166,Camden 001,E01000912,8.82,10.26,66.75,14.17,E01000912,E01000912
2,100054.0,4611.0,The Camden School for Girls,75.0,107.0,38.0,36.0,1025,879.0,85.76,...,184659.0,E02000180,Camden 015,E01035707,9.01,6.20,73.25,11.54,E01035707,E01000864
3,100054.0,4611.0,The Camden School for Girls,75.0,107.0,38.0,36.0,1025,879.0,85.76,...,184659.0,E02000180,Camden 015,E01035707,9.01,6.20,73.25,11.54,E01035707,E01000865
4,100056.0,4688.0,William Ellis School,41.0,111.0,58.0,52.0,836,NaN,NaN,...,186049.0,E02000166,Camden 001,E01000912,8.82,10.26,66.75,14.17,E01000912,E01000912


In [ ]:
idaci_col = 'Income Deprivation Affecting Children Index (IDACI) Decile (where 1 is most deprived 10% of LSOAs)'

# --- 2016/2017 (2011 LSOA) ---
schoolData20162017 = schoolData20162017.merge(
    idaci20162017[['LSOA code (2011)', idaci_col]],
    left_on='LSOA11CD', right_on='LSOA code (2011)', how='left'
).rename(columns={idaci_col: 'IDACI'})
schoolData20162017['IDACI'] = schoolData20162017.groupby('MSOA (code)')['IDACI'].transform('mean').round(2)

# --- 2018/2019 (2011 LSOA) ---
schoolData20182019 = schoolData20182019.merge(
    idaci20182019[['LSOA code (2011)', idaci_col]],
    left_on='LSOA11CD', right_on='LSOA code (2011)', how='left'
).rename(columns={idaci_col: 'IDACI'})
schoolData20182019['IDACI'] = schoolData20182019.groupby('MSOA (code)')['IDACI'].transform('mean').round(2)

# --- 2021/2022 (2011 LSOA) ---
schoolData20212022 = schoolData20212022.merge(
    idaci20212022[['LSOA code (2011)', idaci_col]],
    left_on='LSOA11CD', right_on='LSOA code (2011)', how='left'
).rename(columns={idaci_col: 'IDACI'})
schoolData20212022['IDACI'] = schoolData20212022.groupby('MSOA (code)')['IDACI'].transform('mean').round(2)

# --- 2022/2023 (2021 LSOA) ---
schoolData20222023 = schoolData20222023.merge(
    idaci20222023[['LSOA code (2021)', idaci_col]],
    left_on='LSOA (code)', right_on='LSOA code (2021)', how='left'
).rename(columns={idaci_col: 'IDACI'})
schoolData20222023['IDACI'] = schoolData20222023.groupby('MSOA (code)')['IDACI'].transform('mean').round(2)

# --- 2024/2025 (2021 LSOA) ---
schoolData20242025 = schoolData20242025.merge(
    idaci20242025[['LSOA code (2021)', idaci_col]],
    left_on='LSOA (code)', right_on='LSOA code (2021)', how='left'
).rename(columns={idaci_col: 'IDACI'})
schoolData20242025['IDACI'] = schoolData20242025.groupby('MSOA (code)')['IDACI'].transform('mean').round(2)

In [ ]:
schoolData20242025.head()

,URN,EstablishmentNumber,EstablishmentName,StudentsAchievingGrade5PlusEngMathRate,NumberOfPupils,StudentReceivingFreeSchoolMeals,PercentOfStudentReceivingFreeSchoolMeals,TotalPupils,FemalePupils,PercentFemale,...,Micropolitan,Noncore,AsianPopulationRate,BlackPopulationRate,WhitePopulationRate,OtherRacePopulationRate,LSOA21CD,LSOA11CD,LSOA code (2021),IDACI
0,100053.0,4285.0,Acland Burghley School,52.0,179,92,51.4,1218,440.0,36.12,...,0,0,8.55,4.61,75.44,11.40,E01000928,E01000928,E01000928,7.0
1,100054.0,4611.0,The Camden School for Girls,78.6,117,45,38.5,1076,913.0,84.85,...,0,0,9.01,6.20,73.25,11.54,E01035707,E01000864,E01035707,5.0
2,100054.0,4611.0,The Camden School for Girls,78.6,117,45,38.5,1076,913.0,84.85,...,0,0,9.01,6.20,73.25,11.54,E01035707,E01000865,E01035707,5.0
3,100052.0,4275.0,Hampstead School,42.6,202,111,55.0,1228,603.0,49.10,...,0,0,15.84,5.02,64.03,15.11,E01000871,E01000871,E01000871,7.0
4,100049.0,4104.0,Haverstock School,37.0,146,117,80.1,909,399.0,43.89,...,0,0,19.35,17.16,49.79,13.70,E01000902,E01000902,E01000902,1.0


## **SAMHI**

In [ ]:
samhi20162017 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2016-2017/samhi.csv', encoding="latin-1", low_memory=False)
samhi20182019 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2018-2019/samhi.csv', encoding="latin-1", low_memory=False)
samhi20212022 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2021-2022/samhi.csv', encoding="latin-1", low_memory=False)
samhi20222023 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2022-2023/samhi.csv', encoding="latin-1", low_memory=False)
samhi20242025 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentVariables/2024-2025/samhi.csv', encoding="latin-1", low_memory=False)

In [ ]:
# each year has a different samhi_dec column, so rename to 'SAMHI' after merging

# --- 2016/2017 ---
schoolData20162017 = schoolData20162017.merge(
    samhi20162017[['lsoa11', 'samhi_dec.2016']], left_on='LSOA11CD', right_on='lsoa11', how='left'
).rename(columns={'samhi_dec.2016': 'SAMHI'})
schoolData20162017['SAMHI'] = schoolData20162017.groupby('MSOA (code)')['SAMHI'].transform('mean').round(2)

# --- 2018/2019 ---
schoolData20182019 = schoolData20182019.merge(
    samhi20182019[['lsoa11', 'samhi_dec.2018']], left_on='LSOA11CD', right_on='lsoa11', how='left'
).rename(columns={'samhi_dec.2018': 'SAMHI'})
schoolData20182019['SAMHI'] = schoolData20182019.groupby('MSOA (code)')['SAMHI'].transform('mean').round(2)

# --- 2021/2022 ---
schoolData20212022 = schoolData20212022.merge(
    samhi20212022[['lsoa11', 'samhi_dec.2021']], left_on='LSOA11CD', right_on='lsoa11', how='left'
).rename(columns={'samhi_dec.2021': 'SAMHI'})
schoolData20212022['SAMHI'] = schoolData20212022.groupby('MSOA (code)')['SAMHI'].transform('mean').round(2)

# --- 2022/2023 ---
schoolData20222023 = schoolData20222023.merge(
    samhi20222023[['lsoa11', 'samhi_dec.2022']], left_on='LSOA11CD', right_on='lsoa11', how='left'
).rename(columns={'samhi_dec.2022': 'SAMHI'})
schoolData20222023['SAMHI'] = schoolData20222023.groupby('MSOA (code)')['SAMHI'].transform('mean').round(2)

# --- 2024/2025 (uses 2022 SAMHI, closest available) ---
schoolData20242025 = schoolData20242025.merge(
    samhi20242025[['lsoa11', 'samhi_dec.2022']], left_on='LSOA11CD', right_on='lsoa11', how='left'
).rename(columns={'samhi_dec.2022': 'SAMHI'})
schoolData20242025['SAMHI'] = schoolData20242025.groupby('MSOA (code)')['SAMHI'].transform('mean').round(2)

In [ ]:
schoolData20242025.head()

,URN,EstablishmentNumber,EstablishmentName,StudentsAchievingGrade5PlusEngMathRate,NumberOfPupils,StudentReceivingFreeSchoolMeals,PercentOfStudentReceivingFreeSchoolMeals,TotalPupils,FemalePupils,PercentFemale,...,AsianPopulationRate,BlackPopulationRate,WhitePopulationRate,OtherRacePopulationRate,LSOA21CD,LSOA11CD,LSOA code (2021),IDACI,lsoa11,SAMHI
0,100053.0,4285.0,Acland Burghley School,52.0,179,92,51.4,1218,440.0,36.12,...,8.55,4.61,75.44,11.40,E01000928,E01000928,E01000928,7.0,E01000928,2.0
1,100054.0,4611.0,The Camden School for Girls,78.6,117,45,38.5,1076,913.0,84.85,...,9.01,6.20,73.25,11.54,E01035707,E01000864,E01035707,5.0,E01000864,1.5
2,100054.0,4611.0,The Camden School for Girls,78.6,117,45,38.5,1076,913.0,84.85,...,9.01,6.20,73.25,11.54,E01035707,E01000865,E01035707,5.0,E01000865,1.5
3,100052.0,4275.0,Hampstead School,42.6,202,111,55.0,1228,603.0,49.10,...,15.84,5.02,64.03,15.11,E01000871,E01000871,E01000871,7.0,E01000871,1.0
4,100049.0,4104.0,Haverstock School,37.0,146,117,80.1,909,399.0,43.89,...,19.35,17.16,49.79,13.70,E01000902,E01000902,E01000902,1.0,E01000902,9.0


In [ ]:
schoolData20162017 = schoolData20162017.drop(columns=['lsoa11'])
schoolData20182019 = schoolData20182019.drop(columns=['lsoa11'])
schoolData20212022 = schoolData20212022.drop(columns=['lsoa11'])
schoolData20222023 = schoolData20222023.drop(columns=['lsoa11'])
schoolData20242025 = schoolData20242025.drop(columns=['lsoa11'])

In [ ]:
schoolData20162017.head()

,URN,EstablishmentNumber,EstablishmentName,StudentsAchievingGrade5PlusEngMathRate,NumberOfPupils,StudentReceivingFreeSchoolMeals,PercentOfStudentReceivingFreeSchoolMeals,TotalPupils,FemalePupils,PercentFemale,...,Noncore,AsianPopulationRate,BlackPopulationRate,WhitePopulationRate,OtherRacePopulationRate,LSOA21CD,LSOA11CD,LSOA code (2011),IDACI,SAMHI
0,100055.0,4652.0,Maria Fidelis Roman Catholic Convent School FCJ,40.0,78.0,48.0,62.0,642.0,457.0,71.18,...,0,29.65,16.03,42.44,11.87,E01000955,E01000955,E01000955,1.5,6.5
1,100051.0,4196.0,Regent High School,34.0,110.0,68.0,62.0,881.0,395.0,44.84,...,0,29.65,16.03,42.44,11.87,E01000952,E01000952,E01000952,1.5,6.5
2,100056.0,4688.0,William Ellis School,56.0,117.0,67.0,57.0,805.0,NaN,NaN,...,0,8.82,10.26,66.75,14.17,E01000912,E01000912,E01000912,2.0,4.0
3,100052.0,4275.0,Hampstead School,44.0,201.0,103.0,51.0,1253.0,578.0,46.13,...,0,15.84,5.02,64.03,15.11,E01000871,E01000871,E01000871,5.0,2.0
4,100049.0,4104.0,Haverstock School,31.0,199.0,125.0,63.0,1147.0,472.0,41.15,...,0,19.35,17.16,49.79,13.70,E01000902,E01000902,E01000902,1.0,8.0


In [ ]:
schoolData20162017.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/SecondarySchools/schoolData20162017.csv", index=False)
schoolData20182019.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/SecondarySchools/schoolData20182019.csv", index=False)
schoolData20212022.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/SecondarySchools/schoolData20212022.csv", index=False)
schoolData20222023.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/SecondarySchools/schoolData20222023.csv", index=False)
schoolData20242025.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/SecondarySchools/schoolData20242025.csv", index=False)

##**BORDA RANK/YP SATURATION SCORE**

In [ ]:
borda20162017 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/BordaRanksByYear/finalDf20162017.csv', encoding="latin-1", low_memory=False)
borda20182019 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/BordaRanksByYear/finalDf20182019.csv', encoding="latin-1", low_memory=False)
borda20212022 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/BordaRanksByYear/finalDf20212022.csv', encoding="latin-1", low_memory=False)
borda20222023 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/BordaRanksByYear/finalDf20222023.csv', encoding="latin-1", low_memory=False)
borda20242025 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/BordaRanksByYear/finalDf20242025.csv', encoding="latin-1", low_memory=False)

In [ ]:
borda20162017.columns.tolist()

['URN',
 'EstablishmentName',
 'StudentsAchievingGrade5PlusEngMathRate',
 'borda_rank',
 'borda_points_total',
 'NumberOfPupils',
 'PercentOfStudentReceivingFreeSchoolMeals',
 'UrbanRuralClassification',
 'GOR',
 'LA (code)',
 'LA (name)',
 'LSOA (code)',
 'MSOA (code)',
 'Postcode',
 'longitude',
 'latitude',
 'Easting',
 'Northing']

In [ ]:
schoolData20162017 = schoolData20162017.merge(borda20162017[['URN', 'borda_rank', 'borda_points_total', 'Easting', 'Northing']], on='URN', how='left')
schoolData20182019 = schoolData20182019.merge(borda20182019[['URN', 'borda_rank', 'borda_points_total', 'Easting', 'Northing']], on='URN', how='left')
schoolData20212022 = schoolData20212022.merge(borda20212022[['URN', 'borda_rank', 'borda_points_total', 'Easting', 'Northing']], on='URN', how='left')
schoolData20222023 = schoolData20222023.merge(borda20222023[['URN', 'borda_rank', 'borda_points_total', 'Easting', 'Northing']], on='URN', how='left')
schoolData20242025 = schoolData20242025.merge(borda20242025[['URN', 'borda_rank', 'borda_points_total', 'Easting', 'Northing']], on='URN', how='left')

In [ ]:
schoolData20162017.head()

,URN,EstablishmentNumber,EstablishmentName,StudentsAchievingGrade5PlusEngMathRate,NumberOfPupils,StudentReceivingFreeSchoolMeals,PercentOfStudentReceivingFreeSchoolMeals,TotalPupils,FemalePupils,PercentFemale,...,OtherRacePopulationRate,LSOA21CD,LSOA11CD,LSOA code (2011),IDACI,SAMHI,borda_rank,borda_points_total,Easting_y,Northing_y
0,100055.0,4652.0,Maria Fidelis Roman Catholic Convent School FCJ,40.0,78.0,48.0,62.0,642.0,457.0,71.18,...,11.87,E01000955,E01000955,E01000955,1.5,6.5,4.0,9948.0,529642.0,182873.0
1,100051.0,4196.0,Regent High School,34.0,110.0,68.0,62.0,881.0,395.0,44.84,...,11.87,E01000952,E01000952,E01000952,1.5,6.5,30.0,9903.0,529555.0,183356.0
2,100056.0,4688.0,William Ellis School,56.0,117.0,67.0,57.0,805.0,NaN,NaN,...,14.17,E01000912,E01000912,E01000912,2.0,4.0,398.0,9145.0,528267.0,186049.0
3,100052.0,4275.0,Hampstead School,44.0,201.0,103.0,51.0,1253.0,578.0,46.13,...,15.11,E01000871,E01000871,E01000871,5.0,2.0,211.0,9497.0,524402.0,185633.0
4,100049.0,4104.0,Haverstock School,31.0,199.0,125.0,63.0,1147.0,472.0,41.15,...,13.70,E01000902,E01000902,E01000902,1.0,8.0,43.0,9870.0,528159.0,184498.0


In [ ]:
schoolData20162017 = schoolData20162017.rename(columns={'borda_rank': 'YPAccessibilityRank'})
schoolData20182019 = schoolData20182019.rename(columns={'borda_rank': 'YPAccessibilityRank'})
schoolData20212022 = schoolData20212022.rename(columns={'borda_rank': 'YPAccessibilityRank'})
schoolData20222023 = schoolData20222023.rename(columns={'borda_rank': 'YPAccessibilityRank'})
schoolData20242025 = schoolData20242025.rename(columns={'borda_rank': 'YPAccessibilityRank'})

In [ ]:
for df in [schoolData20162017, schoolData20182019, schoolData20212022,
           schoolData20222023, schoolData20242025]:
    df['YPAccessibilityIndex'] = pd.qcut(
        df['YPAccessibilityRank'],
        q=10,
        labels=False,
        duplicates='drop'
    ) + 1

In [ ]:
schoolData20162017.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20162017.csv", index=False)
schoolData20182019.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20182019.csv", index=False)
schoolData20212022.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20212022.csv", index=False)
schoolData20222023.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20222023.csv", index=False)
schoolData20242025.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20242025.csv", index=False)

In [ ]:
for name in ['schoolData20162017', 'schoolData20182019', 'schoolData20212022',
             'schoolData20222023', 'schoolData20242025']:
    globals()[name] = globals()[name].drop_duplicates()

In [ ]:
schoolData20162017 = schoolData20162017.rename(columns={'Easting_x': 'Easting'})
schoolData20162017 = schoolData20162017.rename(columns={'Northing_x': 'Northing'})

schoolData20182019 = schoolData20182019.drop_duplicates(subset='URN')
schoolData20182019 = schoolData20182019.rename(columns={'Easting_x': 'Easting'})
schoolData20182019 = schoolData20182019.rename(columns={'Northing_x': 'Northing'})

schoolData20212022 = schoolData20212022.drop_duplicates(subset='URN')
schoolData20212022 = schoolData20212022.rename(columns={'Easting_x': 'Easting'})
schoolData20212022 = schoolData20212022.rename(columns={'Northing_x': 'Northing'})

schoolData20222023 = schoolData20222023.drop_duplicates(subset='URN')
schoolData20222023 = schoolData20222023.rename(columns={'Easting_x': 'Easting'})
schoolData20222023 = schoolData20222023.rename(columns={'Northing_x': 'Northing'})

schoolData20242025 = schoolData20242025.drop_duplicates(subset='URN')
schoolData20242025 = schoolData20242025.rename(columns={'Easting_x': 'Easting'})
schoolData20242025 = schoolData20242025.rename(columns={'Northing_x': 'Northing'})

In [ ]:
schoolData20162017 = schoolData20162017.drop(columns=['Easting_y', 'Northing_y'])
schoolData20182019 = schoolData20182019.drop(columns=['Easting_y', 'Northing_y'])
schoolData20212022 = schoolData20212022.drop(columns=['Easting_y', 'Northing_y'])
schoolData20222023 = schoolData20222023.drop(columns=['Easting_y', 'Northing_y'])
schoolData20242025 = schoolData20242025.drop(columns=['Easting_y', 'Northing_y'])

In [ ]:
schoolData20162017.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20162017.csv", index=False)
schoolData20182019.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20182019.csv", index=False)
schoolData20212022.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20212022.csv", index=False)
schoolData20222023.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20222023.csv", index=False)
schoolData20242025.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20242025.csv", index=False)

## **GROUP BY MSOA**

In [ ]:
schoolData20162017 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20162017.csv', encoding="latin-1", low_memory=False)
schoolData20182019 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20182019.csv', encoding="latin-1", low_memory=False)
schoolData20212022 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20212022.csv', encoding="latin-1", low_memory=False)
schoolData20222023 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20222023.csv', encoding="latin-1", low_memory=False)
schoolData20242025 = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20242025.csv', encoding="latin-1", low_memory=False)

In [ ]:
lsoatomsoa = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/MSOA/lsoatomsoa.csv', encoding="latin-1", low_memory=False)

In [ ]:
# --- rebuild the clean LSOA -> MSOA lookup (from lsoatomsoa) ---
lsoa_to_msoa_full = lsoatomsoa[['lsoa21cd', 'msoa21cd']].drop_duplicates()
lsoa_to_msoa_full = lsoa_to_msoa_full.rename(columns={'lsoa21cd': 'LSOA (code)', 'msoa21cd': 'MSOA (code)'})

lsoa_to_msoa_full = lsoa_to_msoa_full.dropna(subset=['LSOA (code)', 'MSOA (code)'])
lsoa_to_msoa_full = lsoa_to_msoa_full[~lsoa_to_msoa_full['MSOA (code)'].str.startswith('N9999', na=False)]
lsoa_to_msoa_full = lsoa_to_msoa_full.drop_duplicates(subset='LSOA (code)', keep='first')
lsoa_to_msoa_full = lsoa_to_msoa_full[lsoa_to_msoa_full['LSOA (code)'].str.startswith('E')]


In [ ]:
# Clean Race df + LSOA-level rates
def broad_group(cat):
    cat = str(cat)
    if cat.startswith('Asian'):
        return 'Asian'
    elif cat.startswith('Black'):
        return 'Black'
    elif cat.startswith('White'):
        return 'White'
    elif cat.startswith('Mixed') or cat.startswith('Other ethnic group'):
        return 'Other'
    else:
        return 'Exclude'

race['BroadGroup'] = race['Ethnic group (20 categories)'].apply(broad_group)
race = race[race['BroadGroup'] != 'Exclude']

lsoa_col = 'Lower layer Super Output Areas Code'

totals = race.groupby(lsoa_col)['Observation'].sum().rename('Total')
wide = race.pivot_table(index=lsoa_col, columns='BroadGroup', values='Observation', aggfunc='sum', fill_value=0)
race_rates = wide.div(totals, axis=0) * 100
race_rates = race_rates.rename(columns={
    'Asian': 'AsianPopulationRate', 'Black': 'BlackPopulationRate',
    'White': 'WhitePopulationRate', 'Other': 'OtherRacePopulationRate',
}).reset_index().rename(columns={lsoa_col: 'LSOA (code)'})

race_cols = ['AsianPopulationRate', 'BlackPopulationRate', 'WhitePopulationRate', 'OtherRacePopulationRate']
lsoa_totals = totals.rename('LSOA_TotalPop').reset_index().rename(columns={lsoa_col: 'LSOA (code)'})

In [ ]:
# Clean LSOA -> MSOA lookup (ONS, full coverage)
lsoa_to_msoa_full = lsoatomsoa[['lsoa21cd', 'msoa21cd']].drop_duplicates()
lsoa_to_msoa_full = lsoa_to_msoa_full.rename(columns={'lsoa21cd': 'LSOA (code)', 'msoa21cd': 'MSOA (code)'})
lsoa_to_msoa_full = lsoa_to_msoa_full.dropna(subset=['LSOA (code)', 'MSOA (code)'])
lsoa_to_msoa_full = lsoa_to_msoa_full[~lsoa_to_msoa_full['MSOA (code)'].str.startswith('N9999', na=False)]
lsoa_to_msoa_full = lsoa_to_msoa_full.drop_duplicates(subset='LSOA (code)', keep='first')

In [ ]:
# Race weighted to MSOA (static across years)
lsoa_weighted = lsoa_to_msoa_full.merge(race_rates, on='LSOA (code)', how='left').merge(lsoa_totals, on='LSOA (code)', how='left')
for c in race_cols:
    lsoa_weighted[f'{c}_weighted'] = lsoa_weighted[c] * lsoa_weighted['LSOA_TotalPop']
msoa_race_weighted = lsoa_weighted.groupby('MSOA (code)').agg(
    **{f'{c}_sum': (f'{c}_weighted', 'sum') for c in race_cols},
    MSOA_TotalPop=('LSOA_TotalPop', 'sum')
).reset_index()
for c in race_cols:
    msoa_race_weighted[c] = (msoa_race_weighted[f'{c}_sum'] / msoa_race_weighted['MSOA_TotalPop']).round(2)
msoa_race_weighted = msoa_race_weighted[['MSOA (code)'] + race_cols]

In [ ]:
# SAMHI per year (LSOA11-based)
def build_samhi_msoa(samhi_year_df, samhi_col):
    samhi_lookup = samhi_year_df[['lsoa11', samhi_col]].rename(columns={'lsoa11': 'LSOA11CD', samhi_col: 'SAMHI'})
    lsoa21_to_11 = lsoa2011to2022[['LSOA21CD', 'LSOA11CD']].rename(columns={'LSOA21CD': 'LSOA (code)'})
    lsoa11_to_msoa = lsoa_to_msoa_full.merge(lsoa21_to_11, on='LSOA (code)', how='left')
    samhi_by_lsoa = lsoa11_to_msoa.merge(samhi_lookup, on='LSOA11CD', how='left')
    return samhi_by_lsoa.groupby('MSOA (code)')['SAMHI'].mean().round(2).reset_index()

samhi_msoa_by_year = {
    '20162017': build_samhi_msoa(samhi20162017, 'samhi_dec.2016'),
    '20182019': build_samhi_msoa(samhi20182019, 'samhi_dec.2018'),
    '20212022': build_samhi_msoa(samhi20212022, 'samhi_dec.2021'),
    '20222023': build_samhi_msoa(samhi20222023, 'samhi_dec.2022'),
    '20242025': build_samhi_msoa(samhi20242025, 'samhi_dec.2022'),
}

In [ ]:
print(lsoa2011to2022[['LSOA21CD', 'LSOA11CD']].isna().sum())
print(lsoa2011to2022['LSOA11CD'].nunique(), 'unique LSOA11 codes in crosswalk')
print(lsoa_to_msoa_full['LSOA (code)'].nunique(), 'unique LSOA21 codes in lsoa_to_msoa_full')

# does every LSOA21 in lsoa_to_msoa_full actually have a LSOA11CD match?
test_crosswalk = lsoa_to_msoa_full.merge(
    lsoa2011to2022[['LSOA21CD', 'LSOA11CD']].rename(columns={'LSOA21CD': 'LSOA (code)'}),
    on='LSOA (code)', how='left'
)
print(test_crosswalk['LSOA11CD'].isna().sum(), 'of', len(test_crosswalk), 'LSOA21s with no LSOA11 match')

LSOA21CD    0
LSOA11CD    0
dtype: int64
34753 unique LSOA11 codes in crosswalk
42650 unique LSOA21 codes in lsoa_to_msoa_full
6978 of 42774 LSOA21s with no LSOA11 match


In [ ]:
# IDACI per year (mixed LSOA11/LSOA21 boundaries)
def build_idaci_msoa(idaci_year_df, lsoa_year):
    code_col = f'LSOA code ({lsoa_year})'
    idaci_lookup = idaci_year_df[[code_col,
        'Income Deprivation Affecting Children Index (IDACI) Decile (where 1 is most deprived 10% of LSOAs)']].rename(
        columns={code_col: 'LSOA_code_raw',
                 'Income Deprivation Affecting Children Index (IDACI) Decile (where 1 is most deprived 10% of LSOAs)': 'IDACI'}
    )
    if lsoa_year == '2011':
        crosswalk = lsoa2011to2022[['LSOA11CD', 'LSOA21CD']].rename(columns={'LSOA11CD': 'LSOA_code_raw'})
        idaci_lookup = idaci_lookup.merge(crosswalk, on='LSOA_code_raw', how='left')
        idaci_lookup = idaci_lookup.rename(columns={'LSOA21CD': 'LSOA (code)'})
        idaci_lookup = idaci_lookup.groupby('LSOA (code)')['IDACI'].mean().reset_index()
    else:
        idaci_lookup = idaci_lookup.rename(columns={'LSOA_code_raw': 'LSOA (code)'})
    idaci_by_lsoa = lsoa_to_msoa_full.merge(idaci_lookup[['LSOA (code)', 'IDACI']], on='LSOA (code)', how='left')
    return idaci_by_lsoa.groupby('MSOA (code)')['IDACI'].mean().round(2).reset_index()

idaci_msoa_by_year = {
    '20162017': build_idaci_msoa(idaci20162017, '2011'),
    '20182019': build_idaci_msoa(idaci20182019, '2011'),
    '20212022': build_idaci_msoa(idaci20212022, '2011'),
    '20222023': build_idaci_msoa(idaci20222023, '2021'),
    '20242025': build_idaci_msoa(idaci20242025, '2021'),
}

In [ ]:
def build_msoa_agg(df, year):
    msoa_borda = (
        df.groupby('MSOA (code)')['borda_points_total']
        .mean().round(3).reset_index()
        .rename(columns={'borda_points_total': 'MSOA_BordaPointsMean'})
    )
    zero_mask = msoa_borda['MSOA_BordaPointsMean'] == msoa_borda['MSOA_BordaPointsMean'].min()
    na_mask = msoa_borda['MSOA_BordaPointsMean'].isna()
    nonzero_mask = ~zero_mask & ~na_mask
    nonzero_rank = msoa_borda.loc[nonzero_mask, 'MSOA_BordaPointsMean'].rank(method='first')
    nonzero_decile = pd.qcut(nonzero_rank, 9, labels=range(1, 10))
    msoa_borda['BordaAccessibilityDecile'] = pd.NA
    msoa_borda.loc[nonzero_mask, 'BordaAccessibilityDecile'] = 10 - nonzero_decile.astype(int)
    msoa_borda.loc[zero_mask, 'BordaAccessibilityDecile'] = 10
    msoa_borda['BordaAccessibilityDecile'] = msoa_borda['BordaAccessibilityDecile'].astype('Int64')

    msoa_agg = df[['MSOA (code)']].drop_duplicates().reset_index(drop=True)
    msoa_agg = msoa_agg.merge(idaci_msoa_by_year[year], on='MSOA (code)', how='left')
    msoa_agg = msoa_agg.merge(samhi_msoa_by_year[year], on='MSOA (code)', how='left')

    weighted_fsm = df.groupby('MSOA (code)').agg(
        TotalPupils=('NumberOfPupils', 'sum'),
        TotalFSM=('StudentReceivingFreeSchoolMeals', 'sum'),
    ).reset_index()
    weighted_fsm['PercentOfStudentReceivingFreeSchoolMeals'] = (weighted_fsm['TotalFSM'] / weighted_fsm['TotalPupils'] * 100).round(2)
    msoa_agg = msoa_agg.merge(weighted_fsm[['MSOA (code)', 'PercentOfStudentReceivingFreeSchoolMeals']], on='MSOA (code)', how='left')

    weighted_female = df.groupby('MSOA (code)').agg(
        TotalPupilsAll=('TotalPupils', 'sum'),
        TotalFemale=('FemalePupils', 'sum'),
    ).reset_index()
    weighted_female['PercentFemale'] = (weighted_female['TotalFemale'] / weighted_female['TotalPupilsAll'] * 100).round(2)
    msoa_agg = msoa_agg.merge(weighted_female[['MSOA (code)', 'PercentFemale']], on='MSOA (code)', how='left')

    msoa_agg = msoa_agg.merge(msoa_race_weighted, on='MSOA (code)', how='left')

    urban_cols = ['LargeCentralMetro','LargeFringeMetro','MediumMetro','SmallMetro','Micropolitan','Noncore']
    msoa_urban_class = df.groupby('MSOA (code)')[urban_cols].mean().idxmax(axis=1).rename('UrbanRuralClass').reset_index()
    msoa_agg = msoa_agg.merge(msoa_urban_class, on='MSOA (code)', how='left')
    for col in urban_cols:
        msoa_agg[col] = (msoa_agg['UrbanRuralClass'] == col).astype(int)

    msoa_gor = df.groupby('MSOA (code)')['GOR'].first().reset_index()
    msoa_agg = msoa_agg.merge(msoa_gor, on='MSOA (code)', how='left')

    msoa_agg = msoa_agg.merge(msoa_borda[['MSOA (code)', 'BordaAccessibilityDecile']], on='MSOA (code)', how='left')

    msoa_msoaname = df.groupby('MSOA (code)')['MSOA (name)'].first().reset_index()
    msoa_agg = msoa_agg.merge(msoa_msoaname, on='MSOA (code)', how='left')

    df = df.copy()
    df['_gcse_achievers'] = (df['StudentsAchievingGrade5PlusEngMathRate'] / 100) * df['NumberOfPupils']
    weighted_gcse = df.groupby('MSOA (code)').agg(
        TotalPupils_GCSE=('NumberOfPupils', 'sum'),
        TotalAchievers=('_gcse_achievers', 'sum'),
        MSOA_SchoolCount=('URN', 'count'),
    ).reset_index()
    weighted_gcse['MSOA_GCSE_Mean'] = (weighted_gcse['TotalAchievers'] / weighted_gcse['TotalPupils_GCSE'] * 100).round(2)
    msoa_agg = msoa_agg.merge(
        weighted_gcse[['MSOA (code)', 'MSOA_GCSE_Mean', 'MSOA_SchoolCount', 'TotalPupils_GCSE', 'TotalAchievers']],
        on='MSOA (code)', how='left'
    )

    total_pupils_msoa = df.groupby('MSOA (code)')['TotalPupils'].sum().rename('MSOA_TotalPupils').reset_index()
    msoa_agg = msoa_agg.merge(total_pupils_msoa, on='MSOA (code)', how='left')

    return msoa_agg

In [ ]:
# Run for all 5 years
year_dfs = {
    '20162017': schoolData20162017, '20182019': schoolData20182019,
    '20212022': schoolData20212022, '20222023': schoolData20222023,
    '20242025': schoolData20242025,
}
msoa_agg_by_year = {}
for year, df in year_dfs.items():
    msoa_agg_by_year[year] = build_msoa_agg(df, year)
    msoa_agg_by_year[year]['IDACI'] = msoa_agg_by_year[year]['IDACI'].round(0)
    msoa_agg_by_year[year]['SAMHI'] = msoa_agg_by_year[year]['SAMHI'].round(0)

print(msoa_agg_by_year['20162017'].columns.tolist())

['MSOA (code)', 'IDACI', 'SAMHI', 'PercentOfStudentReceivingFreeSchoolMeals', 'PercentFemale', 'AsianPopulationRate', 'BlackPopulationRate', 'WhitePopulationRate', 'OtherRacePopulationRate', 'UrbanRuralClass', 'LargeCentralMetro', 'LargeFringeMetro', 'MediumMetro', 'SmallMetro', 'Micropolitan', 'Noncore', 'GOR', 'BordaAccessibilityDecile', 'MSOA (name)', 'MSOA_GCSE_Mean', 'MSOA_SchoolCount', 'TotalPupils_GCSE', 'TotalAchievers', 'MSOA_TotalPupils']


In [ ]:
msoaAgg20162017 = msoa_agg_by_year['20162017']
msoaAgg20182019 = msoa_agg_by_year['20182019']
msoaAgg20212022 = msoa_agg_by_year['20212022']
msoaAgg20222023 = msoa_agg_by_year['20222023']
msoaAgg20242025 = msoa_agg_by_year['20242025']

In [ ]:
msoaAgg20162017.head()

,MSOA (code),IDACI,SAMHI,PercentOfStudentReceivingFreeSchoolMeals,PercentFemale,AsianPopulationRate,BlackPopulationRate,WhitePopulationRate,OtherRacePopulationRate,UrbanRuralClass,...,Micropolitan,Noncore,GOR,BordaAccessibilityDecile,MSOA (name),MSOA_GCSE_Mean,MSOA_SchoolCount,TotalPupils_GCSE,TotalAchievers,MSOA_TotalPupils
0,E02000187,1.0,6.0,61.70,55.94,32.52,15.92,38.26,13.30,LargeCentralMetro,...,0,0,London,1,Camden 022,36.49,2,188.0,68.60,1523.0
1,E02000166,4.0,4.0,57.44,55.58,6.79,7.17,73.10,12.93,LargeCentralMetro,...,0,0,London,2,Camden 001,54.21,2,289.0,156.68,1909.0
2,E02000170,6.0,2.0,51.24,46.13,13.72,5.22,66.93,14.13,LargeCentralMetro,...,0,0,London,1,Camden 005,44.00,1,201.0,88.44,1253.0
3,E02000177,2.0,5.0,62.81,41.15,17.60,12.58,56.97,12.84,LargeCentralMetro,...,0,0,London,1,Camden 012,31.00,1,199.0,61.69,1147.0
4,E02000180,3.0,3.0,37.27,83.53,16.13,11.55,59.92,12.40,LargeCentralMetro,...,0,0,London,1,Camden 015,80.00,2,220.0,176.00,2028.0


In [ ]:
msoaAgg20162017.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/msoaAgg20162017.csv", index=False)
msoaAgg20182019.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/msoaAgg20182019.csv", index=False)
msoaAgg20212022.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/msoaAgg20212022.csv", index=False)
msoaAgg20222023.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/msoaAgg20222023.csv", index=False)
msoaAgg20242025.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/msoaAgg20242025.csv", index=False)

In [ ]:
schoolData20162017.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20162017.csv", index=False)
schoolData20182019.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20182019.csv", index=False)
schoolData20212022.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20212022.csv", index=False)
schoolData20222023.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20222023.csv", index=False)
schoolData20242025.to_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/FinalDFs/schoolData20242025.csv", index=False)